In [0]:
# Bronze Layer - Load Banking Transactions Data

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Define file paths
csv_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/bronze/bank.csv"
delta_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/bronze/transactions"

# Read CSV file with schema inference
df_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

print("=" * 50)
print("BRONZE LAYER - RAW DATA LOADED")
print("=" * 50)

# Display first 10 rows
print("\nFirst 10 rows:")
df_bronze.show(10, truncate=False)

# Print original schema
print("\nOriginal Schema:")
df_bronze.printSchema()

# Clean column names for Delta (remove spaces and special characters)
for column in df_bronze.columns:
    new_column = column.strip().replace(" ", "_").replace(",", "").replace(";", "").replace("{", "").replace("}", "").replace("(", "").replace(")", "").replace("\n", "").replace("\t", "").replace("=", "")
    df_bronze = df_bronze.withColumnRenamed(column, new_column)

print("\nCleaned Schema (for Delta):")
df_bronze.printSchema()

# Display row count
row_count = df_bronze.count()
print(f"\nTotal Records: {row_count:,}")

# Save as Delta table
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print(f"\n✓ Data saved to Delta format: {delta_path}")
print("\n✓ Bronze Layer Load Complete!")

BRONZE LAYER - RAW DATA LOADED

First 10 rows:
+-------------+---------+--------------------------------+-------+----------+----------------+--------------+--------------+---+
|Account No   |DATE     |TRANSACTION DETAILS             |CHQ.NO.|VALUE DATE| WITHDRAWAL AMT | DEPOSIT AMT  |BALANCE AMT   |.  |
+-------------+---------+--------------------------------+-------+----------+----------------+--------------+--------------+---+
|409000611074'|29-Jun-17|TRF FROM  Indiaforensic SERVICES|NULL   |29-Jun-17 |NULL            | 1,000,000.00 | 1,000,000.00 |.  |
|409000611074'|5-Jul-17 |TRF FROM  Indiaforensic SERVICES|NULL   |5-Jul-17  |NULL            | 1,000,000.00 | 2,000,000.00 |.  |
|409000611074'|18-Jul-17|FDRL/INTERNAL FUND TRANSFE      |NULL   |18-Jul-17 |NULL            | 500,000.00   | 2,500,000.00 |.  |
|409000611074'|1-Aug-17 |TRF FRM  Indiaforensic SERVICES |NULL   |1-Aug-17  |NULL            | 3,000,000.00 | 5,500,000.00 |.  |
|409000611074'|16-Aug-17|FDRL/INTERNAL FUND TRANSF